# 07 — Evaluation (Corrected) | RQ3 / RQ4 / RQ5 / RQ7
Five arms: **Zero-Shot · SFT · PPO-dense · PPO-binary · DPO**.

- No silent checkpoint fallback — missing arms are skipped and reported.
- Full benchmarks: HumanEval **164**, MBPP **500** (`EVAL_N = None`).
- Fixed seed + decoding for every arm.
- Smoke only if `EVAL_N` is set small (e.g. 20). **Do not report smoke as paper results.**


In [ ]:
!pip install -q peft transformers datasets pandas
import os, sys, shutil, random, torch, pandas as pd
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

repo = os.path.abspath(os.getcwd())
GIT_SHA = ''
if os.path.isdir('/kaggle/working') and not os.path.exists(os.path.join(repo, 'src', 'training', 'ppo.py')):
    !rm -rf /kaggle/working/src /kaggle/working/temp_repo
    !git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git /kaggle/working/temp_repo
    !cp -r /kaggle/working/temp_repo/src /kaggle/working/src
    _sha = !git -C /kaggle/working/temp_repo rev-parse HEAD
    GIT_SHA = _sha[0] if _sha else ''
    !rm -rf /kaggle/working/temp_repo
    repo = '/kaggle/working'
sys.path.insert(0, repo)
print(f'Eval repo={repo} commit={GIT_SHA or "unknown"} seed={SEED}')

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from src.evaluation.metrics import extract_eval_test_cases
from src.debugging.debug_loop import agentic_debug_loop

BASE = 'deepseek-ai/deepseek-coder-1.3b-instruct'
he = load_dataset('openai_humaneval', split='test')
mbpp = load_dataset('mbpp', split='test')
print('Benchmarks:', len(he), 'HumanEval /', len(mbpp), 'MBPP')


In [ ]:
# Full paper eval: EVAL_N = None (164 HE + 500 MBPP). Smoke: EVAL_N = 20.
EVAL_N = None
MAX_K = 5
ARMS = [
    ('zero_shot', 'Zero-Shot'),
    ('sft', 'SFT'),
    ('ppo_dense', 'PPO-dense'),
    ('ppo_binary', 'PPO-binary'),
    ('dpo', 'DPO'),
]

def adapter_path(tag):
    p = f'./checkpoints/{tag}/final'
    required = ['adapter_config.json', 'adapter_model.safetensors']
    if tag == 'dpo':
        required.append('dpo_metadata.json')
    if tag.startswith('ppo'):
        required.append('ppo_metadata.json')
    return p if all(os.path.isfile(os.path.join(p, f)) for f in required) else None

def load_variant(tag):
    tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto' if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    if tag != 'zero_shot':
        p = adapter_path(tag)
        if p is None:
            return None, tok
        model = PeftModel.from_pretrained(model, p, is_trainable=False)
    model.eval()
    return model, tok

def subset(ds):
    return ds if EVAL_N is None else ds.select(range(min(EVAL_N, len(ds))))

def evaluate_max_k(model, tokenizer, dataset, max_k=MAX_K, label=''):
    total = len(dataset)
    pass1 = 0
    solved_at = {1: 0, 3: 0, 5: 0}
    torch.manual_seed(SEED)
    random.seed(SEED)
    for i, example in enumerate(dataset):
        tests = extract_eval_test_cases(example)
        if not tests:
            raise ValueError(f'No executable tests for evaluation example {i}')
        problem = example.get('prompt', example.get('question', example.get('text', '')))
        history = agentic_debug_loop(model, tokenizer, problem, tests, K=max_k)
        statuses = [h['result']['status'] for h in history]
        pass1 += int(bool(statuses) and statuses[0] == 'AC')
        for k in (1, 3, 5):
            solved_at[k] += int('AC' in statuses[:min(k, len(statuses))])
        if (i + 1) % 5 == 0 or i + 1 == total:
            print(f'{label} [{i+1}/{total}] Pass@1={pass1/(i+1):.2%} Fix@3={solved_at[3]/(i+1):.2%} Fix@5={solved_at[5]/(i+1):.2%}')
    return {
        'total': total,
        'pass_at_1': pass1 / total if total else 0.0,
        'fix_at_1': solved_at[1] / total if total else 0.0,
        'fix_at_3': solved_at[3] / total if total else 0.0,
        'fix_at_5': solved_at[5] / total if total else 0.0,
    }

rows = []
for tag, label in ARMS:
    model, tok = load_variant(tag)
    if model is None:
        print('SKIPPED:', label, f'checkpoint missing at ./checkpoints/{tag}/final')
        continue
    h = evaluate_max_k(model, tok, subset(he), max_k=MAX_K, label=label + ' HE')
    m = evaluate_max_k(model, tok, subset(mbpp), max_k=MAX_K, label=label + ' MBPP')
    rows.append({'model': label, 'K': 1, 'N_HE': h['total'], 'N_MBPP': m['total'],
                 'HE_Pass@1': h['pass_at_1'], 'HE_Fix@K': h['fix_at_1'],
                 'MBPP_Pass@1': m['pass_at_1'], 'MBPP_Fix@K': m['fix_at_1']})
    for k in (3, 5):
        rows.append({'model': label, 'K': k, 'N_HE': h['total'], 'N_MBPP': m['total'],
                     'HE_Pass@1': h['pass_at_1'], 'HE_Fix@K': h[f'fix_at_{k}'],
                     'MBPP_Pass@1': m['pass_at_1'], 'MBPP_Fix@K': m[f'fix_at_{k}']})
    del model
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('evaluation_results_corrected.csv', index=False)
if len(df):
    assert all(df.groupby('model')['HE_Fix@K'].apply(lambda s: list(s) == sorted(s)))
    assert all(df.groupby('model')['MBPP_Fix@K'].apply(lambda s: list(s) == sorted(s)))
if EVAL_N is None:
    print('FULL evaluation (HumanEval 164 + MBPP 500) complete. Commit:', GIT_SHA or 'unknown')
else:
    print(f'SMOKE only (N={EVAL_N}). Do not report as paper results.')


### Required final runs
- Same seed, decoding, executable tests, and full benchmark subsets for all five arms.
- Only cite results after PPO-dense, PPO-binary, and DPO checkpoints exist and are verified.
- Never reuse earlier simulated / pre-fix percentages.
